In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
import os
from pathlib import Path
from IPython.display import clear_output

In [ ]:
def get_os_agnostic_path(path_str):
    """Safely converts Linux or Windows paths to the current native OS format."""
    if pd.isna(path_str):
        return None
    # Force replacement of both slash types to the native OS separator
    clean_path = str(path_str).replace('\\', os.sep).replace('/', os.sep)
    return Path(clean_path)

In [2]:
MERGED_CSV_PATH = "merged_2.csv"
SUMMARY_CSV_PATH = "summary_report.csv" # Or "summary_report_processed.csv"

In [ ]:
# Load the original HABs dictionary
df_merged = pd.read_csv(MERGED_CSV_PATH, low_memory=False)

# Check if we are starting fresh or resuming
if not SUMMARY_CSV_PATH.endswith("_processed.csv"):
    print(f"Creating new processed tracking file...")
    df_summary = pd.read_csv(SUMMARY_CSV_PATH)
    df_summary['processed'] = 0
    df_summary['selected'] = 0
    
    # Create the new filename
    base_name = os.path.splitext(SUMMARY_CSV_PATH)[0]
    processed_csv_path = f"{base_name}_processed.csv"
    
    # Save it immediately
    df_summary.to_csv(processed_csv_path, index=False)
else:
    print(f"Resuming from existing tracking file...")
    processed_csv_path = SUMMARY_CSV_PATH
    df_summary = pd.read_csv(processed_csv_path)

# Find all unique IDs that haven't been processed yet
unprocessed_ids = df_summary[df_summary['processed'] == 0]['case_id'].unique()

if len(unprocessed_ids) == 0:
    print("🎉 All cases have been processed! No pending annotations.")
else:
    for case_id in unprocessed_ids:
        # Clear the Jupyter cell output for a fresh view
        clear_output(wait=True)
        
        # 1. Fetch Original HAB Data
        orig_row = df_merged[df_merged['ID'] == case_id]
        if orig_row.empty:
            print(f"⚠️ Warning: Case ID {case_id} not found in {MERGED_CSV_PATH}. Marking as processed and skipping.")
            df_summary.loc[df_summary['case_id'] == case_id, 'processed'] = 1
            df_summary.to_csv(processed_csv_path, index=False)
            continue
            
        orig_folder = get_os_agnostic_path(orig_row.iloc[0]['tile_folder_path'])
        orig_img_path = orig_folder / "cyfi_prediction_map.png" if orig_folder else None
        
        # 2. Fetch Candidate Data
        candidates = df_summary[(df_summary['case_id'] == case_id) & (df_summary['processed'] == 0)]
        num_cands = len(candidates)
        cand_indices = candidates.index.tolist()
        
        # 3. Setup Visualization (1 Original + N Candidates)
        fig, axes = plt.subplots(1, num_cands + 1, figsize=(5 * (num_cands + 1), 5))
        
        # Ensure 'axes' is always a list even if there's only 1 subplot total (edge case)
        if num_cands == 0: axes = [axes] 
        
        # Plot Original HAB
        try:
            axes[0].imshow(imread(orig_img_path))
            axes[0].set_title(f"Original HAB (ID: {case_id})", fontweight="bold")
        except Exception:
            axes[0].text(0.5, 0.5, 'Image not found', ha='center', va='center')
            axes[0].set_title(f"Original HAB (ID: {case_id})", fontweight="bold")
        axes[0].axis('off')
        
        # Plot Candidates and Collect Metrics
        cand_details = []
        for i, idx in enumerate(cand_indices):
            cand_row = candidates.loc[idx]
            cand_folder = get_os_agnostic_path(cand_row['output_folder'])
            cand_img_path = cand_folder / "cyfi_prediction_map.png" if cand_folder else None
            
            ax = axes[i + 1]
            try:
                ax.imshow(imread(cand_img_path))
                ax.set_title(f"Candidate {i + 1}", fontweight="bold", color="blue")
            except Exception:
                ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
                ax.set_title(f"Candidate {i + 1}", fontweight="bold", color="blue")
            ax.axis('off')
            
            # Format the text block for this candidate
            details = f"--- Candidate {i + 1} ---\n"
            details += f"Class: {cand_row['Overall_Class']} | Total Index: {cand_row['Total_Index']}\n"
            details += f"Clouds: {cand_row['Cloud_Percentage']}% | Water Px: {cand_row['SCL_Water_Pixels']}\n"
            details += f"CyFi -> High: {cand_row['CyFi_High']}, Mod: {cand_row['CyFi_Moderate']}, Low: {cand_row['CyFi_Low']}\n"
            details += f"CNNs -> res18_scl: {cand_row['res18_scl']} | res18_no_scl: {cand_row['res18_no_scl']} | "
            details += f"convnext_scl: {cand_row['convnext_scl']} | rdnet_no_scl: {cand_row['rdnet_no_scl']}\n"
            cand_details.append(details)
            
        plt.tight_layout()
        plt.show()
        
        # 4. Print Candidate Metrics
        for d in cand_details:
            print(d)
            
        # 5. User Input Parsing
        print("=" * 60)
        print("Enter selected candidates separated by commas (e.g., '1, 3').")
        print("Press ENTER to reject all. Include '0' anywhere to save and EXIT.")
        user_input = input("Selection: ")
        
        selected_nums = []
        exit_flag = False
        
        if user_input.strip() != "":
            parts = user_input.split(",")
            for p in parts:
                p = p.strip()
                if p.isdigit():
                    num = int(p)
                    if num == 0:
                        exit_flag = True
                    elif 1 <= num <= num_cands:
                        selected_nums.append(num)
                        
        # 6. Update DataFrame State
        # Mark all candidates for this ID as processed
        df_summary.loc[cand_indices, 'processed'] = 1
        
        # Mark specific chosen candidates as selected
        for num in selected_nums:
            real_idx = cand_indices[num - 1]
            df_summary.loc[real_idx, 'selected'] = 1
            
        # Immediately save progress to disk
        df_summary.to_csv(processed_csv_path, index=False)
        
        # 7. Check Exit Condition
        if exit_flag:
            clear_output(wait=True)
            remaining = len(df_summary[df_summary['processed'] == 0]['case_id'].unique())
            print(f"🛑 Progress safely saved to {processed_csv_path}.")
            print(f"Remaining unique cases to annotate: {remaining}")
            break

    # Final check if loop completed naturally
    if not exit_flag and len(df_summary[df_summary['processed'] == 0]['case_id'].unique()) == 0:
        clear_output(wait=True)
        print("🎉 All cases have been annotated and saved!")